This will be the user space program to pretrain the ML model to
classify between malicious and normal packets. There are multiple ways
in which an ML model can be deployed and fit in the flow of the packets.

One way is using the XDP_AF sockets and sending data collected about the
packets from kernel space to the user space, where the ML model decides the
fate of the packet, does nothing (for drop) or sends packet to the corresponding
user space application after processing it in the required way. This could
also be implemented in other ways using perf buffer, polling etc.


Another approach would be to implement the ML model within the kernel and that
is the approach currently adopted.

Currently, we are implemented a basic in-kernel very simple (logisticRegression)NN
for classification of the packets. We are going to do quantization-aware-training in
the user space and we will store the quantized weights in a BPF map.

Since the kernel has the following restrictions :
    (1) limitations on the quantity of eBPF instructions and stack space,
    (2) prohibitions on unbounded loops, non-static global variables, variadic functions,
        multi-threaded programming, and floating-point representation, and
    (3) enforcement of array bound checks

We will go for a simple model, which would involve less complex calculations sa well

These weights can be updated as we collect more data and used to train the model
after certain intervals of time. Maybe at the end of every 24 hours. But this feature
will be implemented soon.


In [1]:
import glob
import os
import random
import zipfile

import numpy as np
import pandas as pd

from itertools import combinations
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.ao.quantization import QuantStub, DeQuantStub
from torch.utils.data import TensorDataset, DataLoader

from tqdm import tqdm
from pathlib import Path

# Loading the dataset into a dataframe

In [2]:
# list all csv files only
csv_files = glob.glob('*.{}'.format('csv'))
csv_files

[]

In [3]:
# merging the files

PATH = "dataset"
PATH = "dataset"
PATH = "dataset"
joined_files = os.path.join(PATH, "*.csv")

# A list of all joined files is returned
joined_list = glob.glob(joined_files)

# Finally, the files are joined
df_concat = pd.concat(map(pd.read_csv, joined_list), ignore_index=True)
print(df_concat)

          Destination Port   Flow Duration   Total Fwd Packets  \
0                    54865               3                   2   
1                    55054             109                   1   
2                    55055              52                   1   
3                    46236              34                   1   
4                    54863               3                   2   
...                    ...             ...                 ...   
2830738                 53           32215                   4   
2830739                 53             324                   2   
2830740              58030              82                   2   
2830741                 53         1048635                   6   
2830742                 53           94939                   4   

          Total Backward Packets  Total Length of Fwd Packets  \
0                              0                           12   
1                              1                            6   
2           

In [4]:
df_concat.columns = df_concat.columns.str.strip().str.lower().str.replace(' ', '_', regex=False).str.replace('(', '', regex=False).str.replace(')', '', regex=False)
df_concat.head()

,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [5]:
df_labels = df_concat['label']
df_labels.unique()

array(['BENIGN', 'DDoS', 'PortScan', 'Bot', 'Infiltration',
       'Web Attack � Brute Force', 'Web Attack � XSS',
       'Web Attack � Sql Injection', 'FTP-Patator', 'SSH-Patator',
       'DoS slowloris', 'DoS Slowhttptest', 'DoS Hulk', 'DoS GoldenEye',
       'Heartbleed'], dtype=object)

In [6]:
df_concat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 79 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   destination_port             int64  
 1   flow_duration                int64  
 2   total_fwd_packets            int64  
 3   total_backward_packets       int64  
 4   total_length_of_fwd_packets  int64  
 5   total_length_of_bwd_packets  int64  
 6   fwd_packet_length_max        int64  
 7   fwd_packet_length_min        int64  
 8   fwd_packet_length_mean       float64
 9   fwd_packet_length_std        float64
 10  bwd_packet_length_max        int64  
 11  bwd_packet_length_min        int64  
 12  bwd_packet_length_mean       float64
 13  bwd_packet_length_std        float64
 14  flow_bytes/s                 float64
 15  flow_packets/s               float64
 16  flow_iat_mean                float64
 17  flow_iat_std                 float64
 18  flow_iat_max                 int64  
 19  

In [7]:
df_concat.corr(numeric_only=True)

,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,act_data_pkt_fwd,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min
destination_port,1.000000,-0.151680,-0.004236,-0.003947,0.011145,-0.003082,0.097926,-0.045388,0.140220,0.128861,...,-0.003226,0.000897,-0.035562,-0.043717,-0.051859,-0.023194,-0.112585,0.010399,-0.108185,-0.114614
flow_duration,-0.151680,1.000000,0.020857,0.019670,0.065456,0.016186,0.273308,-0.105230,0.143689,0.234437,...,0.015942,-0.001357,0.189299,0.241060,0.294034,0.121171,0.768034,0.243154,0.779527,0.738328
total_fwd_packets,-0.004236,0.020857,1.000000,0.999070,0.365508,0.996993,0.009358,-0.002989,0.000032,0.001403,...,0.887387,-0.000184,0.039937,0.008329,0.030459,0.041283,0.001820,0.000809,0.001906,0.001670
total_backward_packets,-0.003947,0.019670,0.999070,1.000000,0.359451,0.994429,0.009039,-0.002600,-0.000333,0.001026,...,0.882566,0.000018,0.038963,0.006437,0.028602,0.041278,0.001425,0.000492,0.001456,0.001330
total_length_of_fwd_packets,0.011145,0.065456,0.365508,0.359451,1.000000,0.353762,0.197030,-0.000275,0.185262,0.159787,...,0.407448,-0.001209,0.101084,0.103326,0.126493,0.068325,0.022660,0.027064,0.026079,0.018634
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
active_min,-0.023194,0.121171,0.041283,0.041278,0.068325,0.039069,0.105641,-0.025912,0.081170,0.094164,...,0.031394,-0.006834,0.905862,0.033874,0.584503,1.000000,0.118133,0.038302,0.122651,0.112880
idle_mean,-0.112585,0.768034,0.001820,0.001425,0.022660,0.000809,0.181135,-0.071304,0.127959,0.183139,...,0.000837,-0.000876,0.120171,0.036551,0.088904,0.118133,1.000000,0.150248,0.990387,0.990215
idle_std,0.010399,0.243154,0.000809,0.000492,0.027064,0.000105,0.178091,-0.029951,0.178462,0.191278,...,0.000721,-0.003720,0.070586,0.081435,0.070002,0.038302,0.150248,1.000000,0.283330,0.011609
idle_max,-0.108185,0.779527,0.001906,0.001456,0.026079,0.000797,0.199559,-0.073419,0.148402,0.203304,...,0.000929,-0.001407,0.132700,0.055300,0.102816,0.122651,0.990387,0.283330,1.000000,0.961812


In [8]:
def clean_df(df):
    # Remove the space before each feature names
    df.columns = df.columns.str.strip()
    print('dataset shape', df.shape)

    # This set of feature should have >= 0 values
    num = df._get_numeric_data()
    num[num < 0] = 0

    zero_variance_cols = []
    for col in df.columns:
        if len(df[col].unique()) == 1:
            zero_variance_cols.append(col)
    df.drop(zero_variance_cols, axis = 1, inplace = True)
    print('zero variance columns', zero_variance_cols, 'dropped')
    print('shape after removing zero variance columns:', df.shape)

    df.replace([np.inf, -np.inf], np.nan, inplace = True)
    print(df.isna().any(axis = 1).sum(), 'rows dropped')
    df.dropna(inplace = True)
    print('shape after removing nan:', df.shape)

    # Drop duplicate rows
    df.drop_duplicates(inplace = True)
    print('shape after dropping duplicates:', df.shape)

    column_pairs = [(i, j) for i, j in combinations(df, 2) if df[i].equals(df[j])]
    ide_cols = []
    for column_pair in column_pairs:
        ide_cols.append(column_pair[1])
    df.drop(ide_cols, axis = 1, inplace = True)
    print('columns which have identical values', column_pairs, 'dropped')
    print('shape after removing identical value columns:', df.shape)
    return df
df_concat = clean_df(df_concat)

dataset shape (2830743, 79)
zero variance columns ['bwd_psh_flags', 'bwd_urg_flags', 'fwd_avg_bytes/bulk', 'fwd_avg_packets/bulk', 'fwd_avg_bulk_rate', 'bwd_avg_bytes/bulk', 'bwd_avg_packets/bulk', 'bwd_avg_bulk_rate'] dropped
shape after removing zero variance columns: (2830743, 71)
2867 rows dropped
shape after removing nan: (2827876, 71)
shape after dropping duplicates: (2520798, 71)
columns which have identical values [('total_fwd_packets', 'subflow_fwd_packets'), ('total_backward_packets', 'subflow_bwd_packets'), ('fwd_psh_flags', 'syn_flag_count'), ('fwd_urg_flags', 'cwe_flag_count'), ('fwd_header_length', 'fwd_header_length.1')] dropped
shape after removing identical value columns: (2520798, 66)


In [9]:
df_concat.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2520798 entries, 0 to 2830742
Data columns (total 66 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   destination_port             int64  
 1   flow_duration                int64  
 2   total_fwd_packets            int64  
 3   total_backward_packets       int64  
 4   total_length_of_fwd_packets  int64  
 5   total_length_of_bwd_packets  int64  
 6   fwd_packet_length_max        int64  
 7   fwd_packet_length_min        int64  
 8   fwd_packet_length_mean       float64
 9   fwd_packet_length_std        float64
 10  bwd_packet_length_max        int64  
 11  bwd_packet_length_min        int64  
 12  bwd_packet_length_mean       float64
 13  bwd_packet_length_std        float64
 14  flow_bytes/s                 float64
 15  flow_packets/s               float64
 16  flow_iat_mean                float64
 17  flow_iat_std                 float64
 18  flow_iat_max                 int64  
 19  flow_

In [10]:
unique_vals = df_concat['label'].unique()
df_concat['label'].replace(to_replace=unique_vals,
           value= list(range(len(unique_vals))),
           inplace=True)

C:\Users\rushi\AppData\Local\Temp\ipykernel_14728\3066871665.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_concat['label'].replace(to_replace=unique_vals,
C:\Users\rushi\AppData\Local\Temp\ipykernel_14728\3066871665.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_concat['label'].repl

In [11]:
mask = df_concat['label'] != 0
df_concat.loc[mask, 'label'] = 1

In [12]:
df_concat.describe()

,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
count,2.520798e+06,2.520798e+06,2.520798e+06,2.520798e+06,2.520798e+06,2.520798e+06,2.520798e+06,2.520798e+06,2.520798e+06,2.520798e+06,...,2.520798e+06,2.520798e+06,2.520798e+06,2.520798e+06,2.520798e+06,2.520798e+06,2.520798e+06,2.520798e+06,2.520798e+06,2.520798e+06
mean,8.690590e+03,1.659161e+07,1.028174e+01,1.157280e+01,6.119477e+02,1.814440e+04,2.312292e+02,1.920349e+01,6.350497e+01,7.732347e+01,...,2.588550e+01,9.157847e+04,4.619177e+04,1.720171e+05,6.546359e+04,9.337367e+06,5.657941e+05,9.763770e+06,8.892671e+06,1.688914e-01
std,1.901280e+04,3.523276e+07,7.944201e+02,1.056922e+03,1.058827e+04,2.398177e+06,7.563755e+02,6.079834e+01,1.955526e+02,2.968814e+02,...,6.525341e+00,6.866503e+05,4.165844e+05,1.085571e+06,6.111585e+05,2.484818e+07,4.874169e+06,2.561746e+07,2.458143e+07,3.746560e-01
min,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,5.300000e+01,2.080000e+02,2.000000e+00,1.000000e+00,1.200000e+01,6.000000e+00,6.000000e+00,0.000000e+00,6.000000e+00,0.000000e+00,...,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,8.000000e+01,5.062200e+04,2.000000e+00,2.000000e+00,6.600000e+01,1.560000e+02,4.000000e+01,2.000000e+00,3.625000e+01,0.000000e+00,...,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,4.430000e+02,5.333340e+06,6.000000e+00,5.000000e+00,3.320000e+02,9.970000e+02,2.020000e+02,3.700000e+01,5.200000e+01,7.419280e+01,...,3.200000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,6.553500e+04,1.200000e+08,2.197590e+05,2.919220e+05,1.290000e+07,6.554530e+08,2.482000e+04,2.325000e+03,5.940857e+03,7.125597e+03,...,1.380000e+02,1.100000e+08,7.420000e+07,1.100000e+08,1.100000e+08,1.200000e+08,7.690000e+07,1.200000e+08,1.200000e+08,1.000000e+00


In [13]:
df_concat.head()

,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0


In [14]:
feature_list = ['destination_port', 'packet_length_mean','packet_length_std','packet_length_variance','average_packet_size','fwd_iat_mean','fwd_iat_std','fwd_iat_max']

In [15]:
X = df_concat[feature_list]
y = df_concat['label']

# 3-way split: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Scale features - CRITICAL for logistic regression convergence
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Val:   {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.0f}%)')
print(f'Test:  {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nClass distribution (train): BENIGN={int((y_train==0).sum())}, MALICIOUS={int((y_train==1).sum())}')

Train: 1764558 samples (70%)
Val:   378120 samples (15%)
Test:  378120 samples (15%)

Class distribution (train): BENIGN=1466728, MALICIOUS=297830


In [16]:
class LogisticRegression(nn.Module):
    def __init__(self, dim_input, dim_output, hidden_size=16, dropout=0.3):
        super(LogisticRegression, self).__init__()
        self.quant = QuantStub()
        self.hidden = nn.Linear(dim_input, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout)
        self.output = nn.Linear(hidden_size, dim_output)
        self.sigmoid = nn.Sigmoid()
        self.dequant = DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.hidden(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.output(x)
        x = self.sigmoid(x)
        x = self.dequant(x)
        return x

print('Model: 8 -> 16 (ReLU + Dropout 0.3) -> 1 (Sigmoid)')

Model: 8 -> 16 (ReLU + Dropout 0.3) -> 1 (Sigmoid)


In [17]:
def evaluate(model, data, criterion):
    loss = 0.0
    with torch.no_grad():
        for (x, y_target) in data:
            y = model(x)
            loss += criterion(y, y_target)
    return loss

In [18]:
_DIM_INPUT = 8   # number of features
_DIM_OUTPUT = 1  # binary classifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

Using: cuda


In [19]:
X_train = torch.tensor(X_train, dtype=torch.float).to(device)
y_train = torch.tensor(y_train.values, dtype=torch.float).to(device)
X_val = torch.tensor(X_val, dtype=torch.float).to(device)
y_val = torch.tensor(y_val.values, dtype=torch.float).to(device)
X_test = torch.tensor(X_test, dtype=torch.float).to(device)
y_test = torch.tensor(y_test.values, dtype=torch.float).to(device)

print(f'Tensors on: {X_train.device}')
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

Tensors on: cuda:0
Train: torch.Size([1764558, 8]), Val: torch.Size([378120, 8]), Test: torch.Size([378120, 8])


In [20]:
# Clean up memory
del df_concat, X, y, X_temp, y_temp
import gc
gc.collect()

127

In [21]:
model = LogisticRegression(_DIM_INPUT, _DIM_OUTPUT).to(device)
print(f'Model on: {next(model.parameters()).device}')

Model on: cuda:0


In [22]:
# Insert min-max observers in the model

model.qconfig = torch.ao.quantization.default_qconfig
model.train()
model_quantized = torch.ao.quantization.prepare_qat(model) # Insert observers
print(model_quantized)

LogisticRegression(
  (quant): QuantStub(
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (hidden): Linear(
    in_features=8, out_features=16, bias=True
    (weight_fake_quant): MinMaxObserver(min_val=inf, max_val=-inf)
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (output): Linear(
    in_features=16, out_features=1, bias=True
    (weight_fake_quant): MinMaxObserver(min_val=inf, max_val=-inf)
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (sigmoid): Sigmoid(
    (activation_post_process): FixedQParamsFakeQuantize(
      fake_quant_enabled=tensor([1], device='cuda:0', dtype=torch.uint8), observer_enabled=tensor([1], device='cuda:0', dtype=torch.uint8), scale=tensor([0.0039], device='cuda:0'), zero_point=tensor([0], device='cuda:0', dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine
 

In [23]:
def train(X_train_t, y_train_t, X_val_t, y_val_t, model, num_epochs=100, batch_size=4096):

    # Handle class imbalance
    num_pos = y_train_t.sum().item()
    num_neg = len(y_train_t) - num_pos
    pos_weight = num_neg / num_pos
    print(f'Class weight: BENIGN=1.0, MALICIOUS={pos_weight:.2f}')

    sample_weights = torch.ones_like(y_train_t)
    sample_weights[y_train_t == 1] = pos_weight

    criterion = torch.nn.BCELoss(reduction='none')

    # Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Warmup + LR decay
    warmup_epochs = 5
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs  # linear warmup
        return 1.0
    warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=7, factor=0.5)

    # DataLoader
    train_dataset = TensorDataset(X_train_t, y_train_t, sample_weights)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Early stopping
    best_val_f1 = 0.0
    patience = 15
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for batch_x, batch_y, batch_w in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}', leave=False):
            optimizer.zero_grad()
            y_pred = model(batch_x).reshape(-1)
            loss_per_sample = criterion(y_pred, batch_y)
            loss = (loss_per_sample * batch_w).mean()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch_x)

        avg_loss = epoch_loss / len(X_train_t)

        # Scheduler step
        if epoch < warmup_epochs:
            warmup_scheduler.step()
        else:
            plateau_scheduler.step(avg_loss)

        current_lr = optimizer.param_groups[0]['lr']

        # Evaluate on VALIDATION set
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t).reshape(-1)
            predicted = (val_pred > 0.5).float()
            correct = (predicted == y_val_t).sum().item()
            val_acc = correct / len(y_val_t) * 100

            tp = ((predicted == 1) & (y_val_t == 1)).sum().item()
            tn = ((predicted == 0) & (y_val_t == 0)).sum().item()
            fp = ((predicted == 1) & (y_val_t == 0)).sum().item()
            fn = ((predicted == 0) & (y_val_t == 1)).sum().item()
            total_pos = (y_val_t == 1).sum().item()
            total_neg = (y_val_t == 0).sum().item()
            f1 = 2*tp / max(2*tp+fp+fn, 1) * 100

        # Early stopping + best model save
        if f1 > best_val_f1:
            best_val_f1 = f1
            patience_counter = 0
            os.makedirs('src', exist_ok=True)
            torch.save(model.state_dict(), './src/best_model.pth')
            marker = ' *best*'
        else:
            patience_counter += 1
            marker = ''

        print(f'Epoch {epoch+1:3d}/{num_epochs} | Loss: {avg_loss:.4f} | LR: {current_lr:.6f} | '
              f'Acc: {val_acc:.2f}% | F1: {f1:.1f}% | '
              f'Recall(mal): {tp/max(total_pos,1)*100:.1f}% | '
              f'Recall(ben): {tn/max(total_neg,1)*100:.1f}%{marker}')

        # Early stopping check
        if patience_counter >= patience:
            print(f'\nEarly stopping at epoch {epoch+1} (no improvement for {patience} epochs)')
            break

    # --- Find optimal threshold on validation set ---
    print('\n--- Finding optimal threshold on validation set ---')
    # Load best model
    model.load_state_dict(torch.load('./src/best_model.pth', map_location=device))
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t).reshape(-1)

    best_t, best_f1_t = 0.5, 0.0
    for t in [i/100 for i in range(20, 80)]:
        predicted = (val_pred > t).float()
        tp = ((predicted == 1) & (y_val_t == 1)).sum().item()
        fp = ((predicted == 1) & (y_val_t == 0)).sum().item()
        fn = ((predicted == 0) & (y_val_t == 1)).sum().item()
        f1 = 2*tp / max(2*tp+fp+fn, 1) * 100
        if f1 > best_f1_t:
            best_f1_t = f1
            best_t = t

    print(f'Optimal threshold: {best_t:.2f} (F1: {best_f1_t:.2f}%)')
    print(f'Best validation F1: {best_val_f1:.2f}%')
    print(f'Best model loaded from: ./src/best_model.pth')

    global THRESHOLD
    THRESHOLD = best_t
    return

THRESHOLD = 0.5
train(X_train, y_train, X_val, y_val, model_quantized, num_epochs=100, batch_size=4096)

Class weight: BENIGN=1.0, MALICIOUS=4.92


Epoch   1/100 | Loss: 1.0864 | LR: 0.000400 | Acc: 87.79% | F1: 62.0% | Recall(mal): 59.0% | Recall(ben): 93.6% *best*


Epoch   2/100 | Loss: 0.8271 | LR: 0.000600 | Acc: 88.06% | F1: 62.2% | Recall(mal): 58.1% | Recall(ben): 94.2% *best*


Epoch   3/100 | Loss: 0.7514 | LR: 0.000800 | Acc: 89.93% | F1: 66.1% | Recall(mal): 58.2% | Recall(ben): 96.4% *best*


Epoch   4/100 | Loss: 0.6784 | LR: 0.001000 | Acc: 90.63% | F1: 69.8% | Recall(mal): 64.1% | Recall(ben): 96.0% *best*


...

Epoch  16/100 | Loss: 0.5911 | LR: 0.001000 | Acc: 86.69% | F1: 70.8% | Recall(mal): 95.4% | Recall(ben): 84.9%


Epoch  17/100 | Loss: 0.5911 | LR: 0.001000 | Acc: 86.70% | F1: 70.8% | Recall(mal): 95.5% | Recall(ben): 84.9%


Epoch  18/100 | Loss: 0.5910 | LR: 0.001000 | Acc: 86.73% | F1: 70.8% | Recall(mal): 95.4% | Recall(ben): 85.0%


Epoch  19/100 | Loss: 0.5913 | LR: 0.000500 | Acc: 86.71% | F1: 70.8% | Recall(mal): 95.5% | Recall(ben): 84.9%


Epoch  20/100 | Loss: 0.5913 | LR: 0.000500 | Acc: 86.70% | F1: 70.8% | Recall(mal): 95.5% | Recall(ben): 84.9%

Early stopping at epoch 20 (no improvement for 15 epochs)

--- Finding optimal threshold on validation set ---
Optimal threshold: 0.50 (F1: 72.29%)
Best validation F1: 72.29%
Best model loaded from: ./src/best_model.pth


In [30]:
# Load best model from local file
model_path = './src/best_model.pth'
THRESHOLD = 0.50  # Optimal threshold found during training

# Rebuild model, load weights (strict=False to handle observer mismatches)
model_loaded = LogisticRegression(_DIM_INPUT, _DIM_OUTPUT).to(device)
model_loaded.load_state_dict(torch.load(model_path, map_location=device), strict=False)

# Replace model_quantized (use as float model - weights are what matter)
model_quantized = model_loaded
model_quantized.eval()

print(f'Model loaded from: {model_path}')
print(f'Threshold: {THRESHOLD}')

Model loaded from: ./src/best_model.pth
Threshold: 0.5


In [31]:
def print_size_of_model(model):
    torch.save(model.state_dict(), "temp_delme.p")
    print('Size (KB):', os.path.getsize("temp_delme.p")/1e3)
    os.remove('temp_delme.p')

In [32]:
print_size_of_model(model_quantized)
print(f'Check statistics of the various layers')
print(model_quantized)

Size (KB): 2.856
Check statistics of the various layers
LogisticRegression(
  (quant): QuantStub()
  (hidden): Linear(in_features=8, out_features=16, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (output): Linear(in_features=16, out_features=1, bias=True)
  (sigmoid): Sigmoid()
  (dequant): DeQuantStub()
)


In [33]:
def acc(model):
    model.eval()
    with torch.no_grad():
        y_pred = model(X_test).reshape(-1)
        predicted = (y_pred > THRESHOLD).float()

        correct = (predicted == y_test).sum().item()
        total = len(y_test)
        total_mal = (y_test == 1).sum().item()
        total_ben = (y_test == 0).sum().item()
        tp = ((predicted == 1) & (y_test == 1)).sum().item()
        tn = ((predicted == 0) & (y_test == 0)).sum().item()
        fp = ((predicted == 1) & (y_test == 0)).sum().item()
        fn = ((predicted == 0) & (y_test == 1)).sum().item()

        print(f'\nFinal Test Results on {total} samples (threshold={THRESHOLD:.2f}):')
        print(f'  Overall Accuracy:    {correct/total*100:.2f}%')
        print(f'  Malicious Recall:    {tp/max(total_mal,1)*100:.2f}% ({tp}/{total_mal})')
        print(f'  Benign Recall:       {tn/max(total_ben,1)*100:.2f}% ({tn}/{total_ben})')
        print(f'  False Positives:     {fp} (benign flagged as malicious)')
        print(f'  False Negatives:     {fn} (malicious missed)')
        print(f'  Precision:           {tp/max(tp+fp,1)*100:.2f}%')
        print(f'  F1 Score:            {2*tp/max(2*tp+fp+fn,1)*100:.2f}%')

acc(model_quantized)


Final Test Results on 378120 samples (threshold=0.50):
  Overall Accuracy:    89.70%
  Malicious Recall:    82.03% (52569/64086)
  Benign Recall:       91.27% (286608/314034)
  False Positives:     27426 (benign flagged as malicious)
  False Negatives:     11517 (malicious missed)
  Precision:           65.72%
  F1 Score:            72.97%


# testing before quantization

In [34]:
acc(model_quantized)


Final Test Results on 378120 samples (threshold=0.50):
  Overall Accuracy:    89.70%
  Malicious Recall:    82.03% (52569/64086)
  Benign Recall:       91.27% (286608/314034)
  False Positives:     27426 (benign flagged as malicious)
  False Negatives:     11517 (malicious missed)
  Precision:           65.72%
  F1 Score:            72.97%


# Quantize the model using the statistics collected

In [37]:
model.eval()
model_quantized = torch.ao.quantization.convert(model_quantized)
print(f'Check statistics of the various layers')
print(model_quantized)

Check statistics of the various layers
LogisticRegression(
  (quant): QuantStub()
  (hidden): Linear(in_features=8, out_features=16, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (output): Linear(in_features=16, out_features=1, bias=True)
  (sigmoid): Sigmoid()
  (dequant): DeQuantStub()
)


# Print weights and size of the model after quantization

In [38]:
print('=== Weights after quantization ===')
print('\n--- Hidden layer (8 -> 16) ---')
try:
    print('Weights:', torch.int_repr(model_quantized.hidden.weight()))
except TypeError:
    print('Weights:', model_quantized.hidden.weight)
try:
    print('Bias:', model_quantized.hidden.bias())
except TypeError:
    print('Bias:', model_quantized.hidden.bias)

print('\n--- Output layer (16 -> 1) ---')
try:
    print('Weights:', torch.int_repr(model_quantized.output.weight()))
except TypeError:
    print('Weights:', model_quantized.output.weight)
try:
    print('Bias:', model_quantized.output.bias())
except TypeError:
    print('Bias:', model_quantized.output.bias)

print('\nSize after quantization')
try:
    print(f'Hidden weight: {model_quantized.hidden.weight().element_size()} bytes per element')
    print(f'Output weight: {model_quantized.output.weight().element_size()} bytes per element')
except TypeError:
    print(f'Hidden weight: {model_quantized.hidden.weight.element_size()} bytes per element')
    print(f'Output weight: {model_quantized.output.weight.element_size()} bytes per element')

=== Weights after quantization ===

--- Hidden layer (8 -> 16) ---
Weights: Parameter containing:
tensor([[ 0.3150,  0.3887, -0.0214, -1.0878,  0.4540,  0.1653, -0.2169, -0.1056],
        [ 0.0683, -1.1732,  0.0774,  0.7023, -1.6822, -0.0681,  0.0529,  0.2051],
        [-0.4153,  0.2084,  0.0036,  0.4961,  0.0059,  0.1250, -0.0448,  0.1533],
        [ 0.0607, -0.1772,  0.0287, -0.1964, -0.7075, -0.0120,  0.3942,  0.0340],
        [ 0.2462,  0.1917,  0.1669, -0.2832,  0.0265, -0.0067, -0.2615,  0.0166],
        [-0.3657,  0.6230,  0.2818,  0.3615,  0.4981, -0.0044,  0.2046, -0.1093],
        [-0.9759,  0.2647,  0.2645,  0.6771,  0.2407,  0.0138, -0.0641,  0.1843],
        [ 0.0689,  0.2582,  0.1817,  0.2342, -0.0531,  0.2061, -0.2383, -0.2364],
        [-0.4922,  0.2335,  0.1065,  0.5818,  0.3236,  0.1049, -0.0106,  0.2422],
        [ 0.5407,  0.2971,  0.0723, -1.1871,  0.2090, -0.0350, -0.1440,  0.3159],
        [-0.3983,  0.5797,  0.4216,  0.5828,  0.4212,  0.1860, -0.0238,  0.0543],


# testing after quantization


In [39]:
acc(model_quantized)


Final Test Results on 378120 samples (threshold=0.50):
  Overall Accuracy:    89.70%
  Malicious Recall:    82.03% (52569/64086)
  Benign Recall:       91.27% (286608/314034)
  False Positives:     27426 (benign flagged as malicious)
  False Negatives:     11517 (malicious missed)
  Precision:           65.72%
  F1 Score:            72.97%


## Saving the model weights


In [40]:
# Save final model weights
os.makedirs('src', exist_ok=True)
torch.save(model.state_dict(), './src/model_weights.pth')

print('Saved: ./src/model_weights.pth (float32)')
print('Saved: ./src/best_model.pth (best validation checkpoint)')

w = torch.load('./src/model_weights.pth', map_location='cpu')
print('\nModel weights:')
print(w)

Saved: ./src/model_weights.pth (float32)
Saved: ./src/best_model.pth (best validation checkpoint)

Model weights:
OrderedDict([('hidden.weight', tensor([[ 0.1654, -0.3181, -0.1295, -0.0424, -0.2923,  0.2569,  0.1012,  0.0144],
        [ 0.3021,  0.0337, -0.0491, -0.2869, -0.3111,  0.1737, -0.0008, -0.1473],
        [ 0.0884,  0.0493, -0.2306,  0.2167, -0.2037,  0.2251, -0.1962,  0.1308],
        [-0.1925,  0.1596,  0.1129, -0.3496, -0.3175,  0.0542,  0.3298, -0.0220],
        [ 0.0813,  0.2214,  0.2800, -0.0787,  0.0543, -0.3182, -0.1706, -0.2040],
        [ 0.0708,  0.3510, -0.0007,  0.0186,  0.1944,  0.0202,  0.1413, -0.1106],
        [-0.3296, -0.1690, -0.1792,  0.1453, -0.2562,  0.0958, -0.0124,  0.3215],
        [ 0.0053,  0.2916,  0.2128,  0.2651, -0.0180,  0.2103, -0.1618, -0.1687],
        [ 0.0529,  0.0845, -0.0961,  0.2955,  0.1232,  0.0198, -0.0431,  0.2855],
        [ 0.0636, -0.1547, -0.1822, -0.2214, -0.0736,  0.1376, -0.1571,  0.2286],
        [ 0.3097,  0.2645,  0.0455,

In [41]:
for i in w:
  print("Key:",i)
  print("Value:",(w[i]))

Key: hidden.weight
Value: tensor([[ 0.1654, -0.3181, -0.1295, -0.0424, -0.2923,  0.2569,  0.1012,  0.0144],
        [ 0.3021,  0.0337, -0.0491, -0.2869, -0.3111,  0.1737, -0.0008, -0.1473],
        [ 0.0884,  0.0493, -0.2306,  0.2167, -0.2037,  0.2251, -0.1962,  0.1308],
        [-0.1925,  0.1596,  0.1129, -0.3496, -0.3175,  0.0542,  0.3298, -0.0220],
        [ 0.0813,  0.2214,  0.2800, -0.0787,  0.0543, -0.3182, -0.1706, -0.2040],
        [ 0.0708,  0.3510, -0.0007,  0.0186,  0.1944,  0.0202,  0.1413, -0.1106],
        [-0.3296, -0.1690, -0.1792,  0.1453, -0.2562,  0.0958, -0.0124,  0.3215],
        [ 0.0053,  0.2916,  0.2128,  0.2651, -0.0180,  0.2103, -0.1618, -0.1687],
        [ 0.0529,  0.0845, -0.0961,  0.2955,  0.1232,  0.0198, -0.0431,  0.2855],
        [ 0.0636, -0.1547, -0.1822, -0.2214, -0.0736,  0.1376, -0.1571,  0.2286],
        [ 0.3097,  0.2645,  0.0455,  0.1392,  0.0453,  0.3145, -0.0917,  0.1532],
        [-0.1532, -0.3003,  0.0537,  0.1133,  0.0216, -0.2491,  0.2085, 

In [48]:
import os

# Export weights AND scaler for eBPF C program
SCALE_FACTOR = 1024  # Use 1024 (10 bits) for better precision

def export_to_c_header(model, scaler, filepath='./src/model_weights.h'):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    
    with open(filepath, 'w') as f:
        f.write('// Auto-generated eBPF model weights & scaler\n')
        f.write(f'#define SCALE_FACTOR {SCALE_FACTOR}\n\n')
        
        # Scaler Mean (8)
        scaler_mean = (scaler.mean_ * SCALE_FACTOR).astype('int64')
        f.write('static const long long scaler_mean[8] = {')
        f.write(', '.join(map(str, scaler_mean)))
        f.write('};\n\n')
        
        # Scaler Standard Deviation (8)
        # Scale std by SCALE_FACTOR to preserve fractional precision
        scaler_std = (scaler.scale_ * SCALE_FACTOR).astype('int64')
        f.write('static const long long scaler_std[8] = {')
        f.write(', '.join(map(str, scaler_std)))
        f.write('};\n\n')
        
        # Hidden Layer Weights (8 -> 16)
        hidden_w = (model.hidden.weight.detach().cpu().numpy() * SCALE_FACTOR).astype('int64')
        f.write('static const long long hidden_weights[16][8] = {\n')
        for row in hidden_w:
            f.write('    {' + ', '.join(map(str, row)) + '},\n')
        f.write('};\n\n')
        
        # Hidden Layer Bias (16)
        hidden_b = (model.hidden.bias.detach().cpu().numpy() * SCALE_FACTOR).astype('int64')
        f.write('static const long long hidden_bias[16] = {')
        f.write(', '.join(map(str, hidden_b)))
        f.write('};\n\n')
        
        # Output Layer Weights (16 -> 1)
        output_w = (model.output.weight.detach().cpu().numpy() * SCALE_FACTOR).astype('int64')
        f.write('static const long long output_weights[16] = {')
        f.write(', '.join(map(str, output_w[0])))
        f.write('};\n\n')
        
        # Output Layer Bias (1)
        output_b = (model.output.bias.detach().cpu().numpy() * SCALE_FACTOR).astype('int64')
        f.write(f'static const long long output_bias = {output_b[0]};\n')
        
        print(f'Successfully exported eBPF weights to {filepath}')

export_to_c_header(model_quantized, scaler)

Successfully exported eBPF weights to ./src/model_weights.h


In [47]:
import glob
import pandas as pd
import numpy as np
import os

# Find a True Positive (Malicious row that the model correctly predicts as > 0.53)
model_quantized.eval()
found = False
joined_files = glob.glob(os.path.join('dataset', '*.csv'))

feature_keys = [
    'destination port', 'packet length mean', 'packet length std',
    'fwd iat mean', 'fwd iat std', 'fwd iat max',
    'flow duration', 'init_win_bytes_forward'
]

for file in joined_files:
    df = pd.read_csv(file)
    # Clean columns
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_', regex=False).str.replace('(', '', regex=False).str.replace(')', '', regex=False)
    
    malicious_df = df[df['label'] != 'BENIGN']
    for idx, row in malicious_df.iterrows():
        features = [float(row[k.replace(' ', '_')]) if pd.notna(row[k.replace(' ', '_')]) else 0.0 for k in feature_keys]
        
        # Scale
        X_scaled = scaler.transform(np.array(features).reshape(1, -1))
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(device)
        
        with torch.no_grad():
            prob = model_quantized(X_tensor).item()
            
        if prob > 0.53: # True Positive
            print(f'True Positive Found! PyTorch Probability: {prob:.4f}')
            print(f'long long ddos_features[8] = {{{", ".join([str(int(f)) for f in features])}}};')
            found = True
            break
    if found:
        break


c:\Users\rushi\Documents\ML_MODEL\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\rushi\Documents\ML_MODEL\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\rushi\Documents\ML_MODEL\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\rushi\Documents\ML_MODEL\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\rushi\Documents\ML_MODEL\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted w

True Positive Found! PyTorch Probability: 0.5734
long long ddos_features[8] = {80, 1454, 4097, 353, 495, 704, 916963, 8192};


c:\Users\rushi\Documents\ML_MODEL\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\rushi\Documents\ML_MODEL\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\rushi\Documents\ML_MODEL\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
